In [ ]:
# use already existent data if its available or query the database if it's not
if not os.path.exists("merged.csv"):

    # %% Define MySQL connection parameters
    db_user = "student"
    db_password = "#q6a21I&OA5k"
    host = "132.252.60.112"
    port = 3306
    dbname = "ENTSOE"

    #  %%Create MySQL engine
    engine = create_engine(
        f"mysql://{urllib.parse.quote_plus(db_user)}:{urllib.parse.quote_plus(db_password)}@{host}:{port}/{dbname}"
    )

    # %% Obtain specification table
    spec_query = "SELECT * FROM spec"
    spec = pd.read_sql_query(spec_query, engine)

    # Get an overview
    print(spec)
    spec.MapTypeCode.unique()
    spec.MapCode.unique()
    spec.Name.unique()
    spec.ProductionType.unique()
    spec.ResolutionCode.unique()
    spec.to_csv("spec.csv")

    # %% Get value by the production types, you can add more
    #  Narrow down the spec table to get the targetTimeSeriesID's
    targets = spec[
        (spec["Name"] == "Generation")
        & (spec["Type"].isin(["DayAhead", "Actual"]))
        & (spec["ProductionType"].isin(["Wind Onshore", "Wind Offshore", "Solar"]))
        & (spec["MapCode"] == "AT")  # Put desired BZN here
        & (spec["MapTypeCode"] == "BZN")
        ]

    # %% Obtain the forecasts table
    values_query = f"""
    SELECT *
    FROM vals
    WHERE TimeSeriesID IN ({", ".join(map(str, targets['TimeSeriesID']))})
    AND YEAR(`DateTime`) >= '2014'
    """
    values = pd.read_sql_query(values_query, engine)

    # %%

    # Get the actual data
    data = pd.merge(values, targets, on="TimeSeriesID")
    data = data[data["DateTime"].dt.year >= 2014]  # You can change the year here

    # Select and wrangle even further
    data = data[["DateTime", "Type", "ProductionType", "Value"]]
    data = data.sort_values(by="DateTime")
    data_generations = data.pivot_table(
        index="DateTime", columns=["Type", "ProductionType"], values="Value"
    )

    print(data_generations)
    data_generations.describe()
    data_generations.to_csv("data_generations.csv")

    # We have used the DE_LU BZN in the earlier example (data_generations).
    # You can get DE_AT_LU or any other zone data, such as FR, PT, and so on.
    # The DE_AT_LU zone was split into DE_LU and AT.
    # Please take a closer look at the specs.
    # To supplement further, you may follow the "R" codes.

    # ************************************************************************
    # %% Load data
    # *************************************************************************

    #  Narrow down the spec table to get the targetTimeSeriesID's
    targets = spec[
        (spec["Name"] == "Load")
        & (spec["Type"].isin(["DayAhead", "Actual"]))
        & (spec["MapCode"] == "AT")  # Put desired BZN here
        & (spec["MapTypeCode"] == "BZN")
        ]

    # %% Obtain the forecasts table
    values_query = f"""
    SELECT *
    FROM vals
    WHERE TimeSeriesID IN ({", ".join(map(str, targets['TimeSeriesID']))})
    AND YEAR(`DateTime`) >= '2014'
    """
    values = pd.read_sql_query(values_query, engine)

    # Get the actual data
    data = pd.merge(values, targets, on="TimeSeriesID")
    data = data[data["DateTime"].dt.year >= 2014]

    # Select and wrangle even further
    data = data[["DateTime", "Type", "Value"]]
    data = data.sort_values(by="DateTime")
    data_load = data.pivot_table(index="DateTime", columns=["Type"], values="Value")

    print(data_load)
    data_load.describe()
    data_load.to_csv("data_load.csv")

    # You can save the data to your desired directory, for further use

    # ************************************************************************
    # %% Priece data
    # *************************************************************************
    #  Narrow down the spec table to get the targetTimeSeriesID's
    targets = spec[
        (spec["Name"] == "Price")
        & (spec["Type"].isin(["DayAhead", "Actual"]))
        # & (spec["ProductionType"].isin(["Wind Onshore", "Wind Offshore", "Solar"]))
        & (spec["MapCode"] == "AT")  # Put desired price zones (BZN)here
        & (spec["MapTypeCode"] == "BZN")
        ]

    # %% Obtain the forecasts table
    values_query = f"""
    SELECT *
    FROM vals
    WHERE YEAR(`DateTime`) >= '2014'AND
    TimeSeriesID IN ({", ".join(map(str, targets['TimeSeriesID']))})"""

    values = pd.read_sql_query(values_query, engine)

    # Get the actual data
    data = pd.merge(values, targets, on="TimeSeriesID")
    data = data[data["DateTime"].dt.year >= 2014]

    # Select and wrangle even further
    data = data[["DateTime", "Type", "Value"]]
    data = data.sort_values(by="DateTime")
    data_price = data.pivot_table(index="DateTime", columns=["Type"], values="Value")

    print(data_price)
    data_price.describe()
    data_price.to_csv("data_price.csv")

    # Merge the dataframes
    # For data_generations (2 column levels: Type + ProductionType)
    # This flattens the MultiIndex columns by joining the two levels with an underscore
    data_generations.columns = ['_'.join(col).strip() for col in data_generations.columns.values]

    # for data_load (only typ as columnname)
    data_load.columns = [f'Load_{col}' for col in data_load.columns]

    # for data_price (only typ as columnname)
    data_price.columns = [f'Price_{col}' for col in data_price.columns]

    # Merge the dataframes on DateTime
    merged = (
        pd.merge(data_generations, data_load, on="DateTime", how="outer")
        .merge(data_price, on="DateTime", how="outer")
    )
    # print and save the data
    print(merged.info())
    print(merged.head())
    merged.to_csv("merged.csv")

else:
    merged = pd.read_csv("merged.csv", parse_dates=["DateTime"])
df = merged
#creature new feature based on the kind of day like workday
at_holidays = holidays.Austria()

df["target"] = df["Load_Actual"].shift(-1)

df["is_weekend"] = (df["DateTime"].dt.weekday >= 5).astype(int)
df["is_holiday"] = df["DateTime"].dt.date.map(at_holidays.__contains__).astype(int)
df["day_of_week"] = df["DateTime"].dt.weekday
df["week_of_year"] = df["DateTime"].dt.isocalendar().week.astype(int) - 1
df['is_workday'] = (~df['is_weekend'] & ~df['is_holiday']).astype(int)

seconds_in_day = 24 * 60 * 60
time_in_seconds = (
        df["DateTime"].dt.hour * 3600
        + df["DateTime"].dt.minute * 60
        + df["DateTime"].dt.second
)
df["time_sin"] = np.sin(2 * np.pi * time_in_seconds / seconds_in_day)
df["time_cos"] = np.cos(2 * np.pi * time_in_seconds / seconds_in_day)

dayahead_cols = df.filter(like="DayAhead").columns
df = df.drop(columns=dayahead_cols)

# delete row pre 1.10.2018
cutoff = pd.Timestamp("2018-10-01")
df = df[df["DateTime"] >= cutoff]

# delete all rows with NaNs
full_rows = ~df.isna().any(axis=1)
last_full_idx = full_rows[full_rows].index[-1]

# Alles danach verwerfen
df = df.loc[:last_full_idx]

# add lags features
df["load_lag_1"] = df["Load_Actual"].shift(1)
df["load_lag_2"] = df["Load_Actual"].shift(2)
df["load_lag_2"] = df["Load_Actual"].shift(3)
df["load_lag_4"] = df["Load_Actual"].shift(4)
df["load_lag_96"] = df["Load_Actual"].shift(96)
df["load_diff_1"] = df["load_lag_1"] - df["load_lag_2"]
df["load_diff_4"] = df["load_lag_1"] - df["load_lag_4"]
df["load_diff_24h"] = df["load_lag_1"] - df["load_lag_96"]

# rolling-statistics
df["load_mean_1h"] = df["Load_Actual"].rolling(4).mean()
df["load_std_1h"] = df["Load_Actual"].rolling(4).std()
df["load_ramp_1h"] = df["Load_Actual"].rolling(4).max() - df["Load_Actual"].rolling(4).min()

# cycling day ot the year
DOY = df['DateTime'].dt.dayofyear
angle = 2 * np.pi * DOY / 365
df['doy_sin'] = np.sin(angle)
df['doy_cos'] = np.cos(angle)

# delete NaNs after feature Engeneering
df = df.dropna().reset_index(drop=True)
# get start and end date for weather Data
start_date = df["DateTime"].min().date()
end_date = df["DateTime"].max().date()

# example: coordinates for Vienna
latitude = 48.2082
longitude = 16.3738

# query weather data via api
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": start_date.isoformat(),
    "end_date": end_date.isoformat(),
    "hourly": "temperature_2m,shortwave_radiation,wind_speed_10m",
    "timezone": "auto"
}

response = requests.get(url, params=params)
data = response.json()

# weather dataframe
hourly_data = data.get("hourly", {})
weather = pd.DataFrame(hourly_data)

# time column in datetime
if "time" in weather.columns:
    weather["time"] = pd.to_datetime(weather["time"])

# format and sort datatime
df = df.sort_values("DateTime")
weather = weather.sort_values("time")

# merge on datetime (left join: df ← weather)
df = pd.merge_asof(df, weather, left_on="DateTime", right_on="time", direction="backward")

# drop weather time
df = df.drop(columns=["time"])

# Parameter for the curvature strength
p = 1.2  # 1 = linear, >1 = stronger increase at extreme values

# Heating demand: stronger increase below 15 °C
df["heating_demand"] = df["temperature_2m"].apply(lambda t: (15 - t) ** p if t < 15 else 0)

# Cooling demand: stronger increase above 22 °C
df["cooling_demand"] = df["temperature_2m"].apply(lambda t: (t - 22) ** p if t > 22 else 0)


In [ ]:
# baseline model: Linear Regression as comparison
features = df.drop(columns=["DateTime", "target"])
target = df["target"]

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

print(f"Linear Regression MAE: {mae:.2f}")
print(f"Linear Regression RMSE: {rmse:.2f}")